# Heavy metals at former pesticide warehouses: what the XRF data show and what the model can do

A demonstration notebook of the pilot project. It shows how the stand's calculations for metals work, on real data
and without a "black box": every chart is built here from the project export.

**Contents**

1. Data: 10 warehouses, 169 field X-ray fluorescence (XRF) points, 9 elements.
2. Comparison with standards: how many points exceed national and international levels.
3. Warehouse priority: which warehouse is more contaminated than the others.
4. The model: where to look for the hotspot within a warehouse, and how this is tested on a warehouse the model has not seen.
5. How much data is needed for the model to become more accurate.
6. Link with POPs pesticide sites.

**How to read.** The code is hidden; the "Show code" button at the top of the page reveals it. All calculations run
locally from the file `tajikistan_soil_ml_export.gpkg`, with no database and no internet.

In [ ]:
import sys
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json
import mltj, demo_utils as U
from demo_utils import f, pct
U.setup('en')
sys.path.insert(0, str(U.ROOT / 'webapp'))
from i18n import name_en                          # English warehouse names
sites, pts, cov = mltj.load()
wgs_sites, wgs_pts = sites.to_crs(4326), pts.to_crs(4326)
print(f"Warehouses: {len(sites)}, XRF points: {len(pts)}, elements: {len(mltj.ELEMENTS)}")

## 1. Data

At each warehouse the XRF points are located inside the warehouse outline and around it. For each point we know the
concentrations of 9 elements (As, Co, Hg, Mo, Pb, Rb, Th, U, Zn), the distance to the warehouse outline, the distance
to water and the soil pH.

In [ ]:
g = pts.groupby('site_name')
t = pd.DataFrame({
    'Warehouse': [name_en(n) for n in g.size().index],
    'Points': g.size().values,
    'Inside outline': g.inside.sum().astype(int).values,
    'Pb, median, mg/kg': g.Pb_mgkg.median().values,
    'Pb, maximum, mg/kg': g.Pb_mgkg.max().values,
}).sort_values('Pb, median, mg/kg', ascending=False)
U.table(t, nd=0)

b = wgs_pts.total_bounds; pad = .35
fig, ax = plt.subplots(figsize=(8.5, 5.6))
U.hillshade(ax, (b[0]-pad, b[1]-pad, b[2]+pad, b[3]+pad))
c = wgs_sites.geometry.centroid
med = g.Pb_mgkg.median().reindex(wgs_sites.name.values).values
sc = ax.scatter(c.x, c.y, s=90 + 3*med, c=med, cmap='YlOrRd', edgecolor='#333', lw=.8, zorder=3, vmin=0)
OFF = {'Турсунзаде': (-12, -16, 'right'), 'Шахринав': (0, 12, 'center'), 'Гиссар': (10, -14, 'left')}  # label offsets by stored name
for (x, y_), n in zip(zip(c.x, c.y), wgs_sites.name):
    dx, dy, ha = OFF.get(n, (9, 4, 'left'))
    ax.annotate(name_en(n), (x, y_), xytext=(dx, dy), textcoords='offset points', fontsize=9, zorder=4, ha=ha,
                bbox=dict(boxstyle='round,pad=.15', fc='white', ec='none', alpha=.75))
plt.colorbar(sc, ax=ax, shrink=.7, label='Pb, warehouse median, mg/kg')
U.map_axes(ax); ax.set_title('10 warehouses with XRF surveys (size and colour: median lead)')
plt.show()

## 2. Comparison with standards

Two comparison levels are used:

- **MPC** (maximum permissible concentration) from the USSR list No. 6229-91, which the client refers to: Pb 32,
  As 2, Hg 2.1 mg/kg; for zinc the list gives only the mobile form, so the APC for total zinc in neutral loams is used
  (220 mg/kg; soil pH at the warehouses is 7.2–7.7);
- **international levels for residential land**: Pb 200 (US EPA, 2025), As 12, Hg 6.6, Zn 250 mg/kg (CCME, Canada).

An important caveat: for portable XRF the detection limits for arsenic and mercury are close to the standards
themselves. Exceedances for As and Hg are a signal for laboratory verification, not a final conclusion. XRF is
reliable for lead and zinc.

In [ ]:
N = {'Pb': (32, 200), 'As': (2, 12), 'Hg': (2.1, 6.6), 'Zn': (220, 250)}
rows = []
for e, (pdk, intl) in N.items():
    v = pts[f'{e}_mgkg']
    rows.append({'Element': e, 'Median, mg/kg': v.median(), 'Maximum, mg/kg': v.max(),
                 'MPC / APC, mg/kg': pdk, 'Above MPC': (v > pdk).mean(),
                 'Intl. level, mg/kg': intl, 'Above intl. level': (v > intl).mean()})
r = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8, 3.6))
x = np.arange(len(r)); w = .38
ax.bar(x - w/2, 100*r['Above MPC'], w, color=U.ORANGE, label='above MPC / APC (USSR)')
ax.bar(x + w/2, 100*r['Above intl. level'], w, color=U.BLUE, label='above international level (residential)')
for i, rr in r.iterrows():
    ax.text(i - w/2, 100*rr['Above MPC'] + 2, pct(rr['Above MPC']), ha='center', fontsize=9)
    ax.text(i + w/2, 100*rr['Above intl. level'] + 2, pct(rr['Above intl. level']), ha='center', fontsize=9)
ax.set_xticks(x, r['Element']); ax.set_ylim(0, 112); ax.set_ylabel('share of points, %')
ax.set_title('Share of the 169 XRF points above the standards'); ax.legend(loc='upper center', bbox_to_anchor=(.5, -.1), ncol=2, fontsize=9)
plt.show()

r2 = r.copy()
r2['Above MPC'] = r2['Above MPC'].map(pct); r2['Above intl. level'] = r2['Above intl. level'].map(pct)
U.table(r2, nd=1)

In [ ]:
pb = pts.Pb_mgkg
U.note(f"Lead: {pct((pb > 32).mean())} of points are above the MPC of 32 mg/kg, but only {int((pb > 200).sum())} of {len(pb)} "
       f"are above the international level of 200 mg/kg. Lead contamination is widespread but moderate; "
       f"arsenic and mercury need laboratory confirmation.")

## 3. Warehouse priority

For each point we compute how many times its Pb, As, Hg and Zn concentrations exceed typical values (medians over
the whole survey) on a logarithmic scale, then average over the four elements. The warehouse index is the median over
its points. The method does not depend on the choice of standard and compares warehouses with each other.

In [ ]:
sub = ['Pb', 'As', 'Hg', 'Zn']
ex = pd.concat([np.log(pts[f'{e}_mgkg']) - np.log(pts[f'{e}_mgkg']).median() for e in sub], axis=1)
pts['composite'] = ex.mean(axis=1)
pr = pts.groupby('site_name').composite.median().sort_values()
fig, ax = plt.subplots(figsize=(7.5, 4))
cols = [U.BLUE if i >= len(pr) - 3 else U.GREY for i in range(len(pr))]
ax.barh([name_en(n) for n in pr.index], pr.values, color=cols)
ax.axvline(0, color='#555', lw=.8)
for i, v in enumerate(pr.values):
    ax.text(v + (.01 if v >= 0 else -.01), i, f(v, 2), va='center', ha='left' if v >= 0 else 'right', fontsize=9)
ax.set_xlim(pr.min() - .14, pr.max() + .12)
ax.set_xlabel('warehouse contamination index (0: typical level for the survey)')
ax.set_title('Warehouse priority for metals: the top three highlighted')
U.comma_axis(ax, 'x', 1)
plt.show()
same = list(pr.index[::-1]) == list(sites.sort_values('priority_rank').name)
U.note(f"Top priority: {', '.join(name_en(n) for n in pr.index[::-1][:3])}. " + ("The order matches the stand ranking."
       if same else "The order differs from the stand ranking; check the export version."))

## 4. The model: where to look for the hotspot within a warehouse

**Task.** Predict in which part of a warehouse lead is higher than usual for that warehouse. The target is the
deviation of a point's log(Pb) from the median of its warehouse: the model thus learns the position of the hotspot,
not differences between warehouses.

**Point features:** distance to the outline (inside or outside), distance to the warehouse centre, zone (inside,
up to 50 m, up to 100 m, beyond), distance to water, soil pH.

**Algorithm:** random forest (300 trees). **Validation:** training on 9 warehouses and prediction for the tenth,
which the model has not seen (leave-one-warehouse-out), in turn for every warehouse.

In [ ]:
y = mltj.target(pts, 'Pb')
res = []
for name, feats in [('point position relative to the warehouse only', mltj.BASE),
                    ('+ distance to water and pH (stand model)', mltj.FEATURES)]:
    r2 = [mltj.lowo_r2(pts, y, feats, seed=s)[0] for s in range(5)]
    res.append({'Feature set': name, 'R², mean': np.mean(r2), 'R², min': np.min(r2), 'R², max': np.max(r2)})
U.table(pd.DataFrame(res), nd=3)
P = np.mean([mltj.lowo_predict(pts, y, mltj.FEATURES, seed=s) for s in range(5)], axis=0)
pts['pred'] = P

An R² of about 0.16 means the model explains roughly one sixth of the differences within warehouses. That is not
enough for a quantitative concentration forecast, but enough to **suggest where to sample first**. Below is an example
of a warehouse the model did not see during training.

In [ ]:
site = pr.index[-1]   # the highest-priority warehouse
S = pts[pts.site_name == site]; poly = sites.loc[S.site_id.iloc[0]].geometry
fig, axs = plt.subplots(1, 2, figsize=(10, 4.6), sharex=True, sharey=True)
for ax, col, title, cmap in [(axs[0], S.Pb_mgkg, f'{name_en(site)}: measured Pb, mg/kg', 'YlOrRd'),
                             (axs[1], S.pred.rank(pct=True), f'{name_en(site)}: model score (rank within warehouse)', 'Blues')]:
    gx, gy = poly.exterior.xy
    ax.fill(gx, gy, color='#eee', ec='#555', lw=1.2, zorder=1)
    sc = ax.scatter(S.geometry.x, S.geometry.y, c=col, cmap=cmap, s=70, edgecolor='#333', lw=.6, zorder=3)
    plt.colorbar(sc, ax=ax, shrink=.75)
    ax.set_title(title, fontsize=10); ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
k = int(np.ceil(len(S) * .25)); top = S.nlargest(k, 'pred')
axs[1].scatter(top.geometry.x, top.geometry.y, s=220, facecolor='none', edgecolor=U.ORANGE, lw=2, zorder=4,
               label='top quarter by the model')
axs[1].legend(loc='upper center', bbox_to_anchor=(.5, -.02), fontsize=8)
plt.suptitle('The model was trained on the other 9 warehouses', fontsize=10, color='#555'); plt.show()

In [ ]:
rows = []
COL = 'Share of the most contaminated points found by the model'
for s, gg in pts.groupby('site_name'):
    k = int(np.ceil(len(gg) * .25))
    top_pred, top_act = set(gg.nlargest(k, 'pred').index), set(gg.nlargest(k, 'Pb_mgkg').index)
    rows.append({'Warehouse': name_en(s), 'Points': len(gg), 'Top quarter, points': k, COL: len(top_pred & top_act) / k})
cap = pd.DataFrame(rows).sort_values(COL, ascending=False)
m = cap[COL].mean()

fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.bar(cap['Warehouse'], 100*cap[COL], color=U.BLUE)
ax.axhline(25, color=U.GREY, ls='--', lw=1.5); ax.text(9.4, 26.5, 'random selection: 25%', ha='right', color='#555', fontsize=9)
ax.axhline(100*m, color=U.BLUE, ls=':', lw=1.5); ax.text(9.4, 100*m + 1.5, f'mean: {pct(m)}', ha='right', color=U.BLUE, fontsize=9)
ax.set_ylabel('%'); ax.set_ylim(0, 100); plt.xticks(rotation=35, ha='right')
ax.set_title('How many of the 25% most contaminated points fall in the model\'s top quarter')
plt.show()
U.note(f"If samples are taken first at the quarter of points flagged by the model, on average {pct(m)} of the warehouse's "
       f"most contaminated points are found; with random selection, 25%. The gain is real but moderate, and it varies between warehouses.")

## 5. How much data is needed

The model is trained on 3, 5, 7 and 9 randomly chosen warehouses and each time tested on a warehouse outside training.
If performance grows with the number of warehouses, the limitation is the amount of data, not the method.

In [ ]:
import random
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
F = mltj.FEATURES; ids = list(pts.site_id.unique()); lc = {}
for k in (3, 5, 7, 9):
    vals = []
    for rep in range(8):
        rnd = random.Random(rep * 100 + k); pred = np.zeros(len(y))
        for te_s in ids:
            tr_s = rnd.sample([s for s in ids if s != te_s], k)
            tr, te = pts.site_id.isin(tr_s).values, (pts.site_id == te_s).values
            pred[te] = RandomForestRegressor(n_estimators=100, min_samples_leaf=3, random_state=rep).fit(
                pts.loc[tr, F].values, y[tr]).predict(pts.loc[te, F].values)
        vals.append(r2_score(y, pred))
    lc[k] = vals
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ks = list(lc); mu = [np.mean(lc[k]) for k in ks]
ax.plot(ks, mu, 'o-', color=U.BLUE, lw=2)
ax.fill_between(ks, [np.percentile(lc[k], 10) for k in ks], [np.percentile(lc[k], 90) for k in ks], color=U.BLUE, alpha=.15)
for k, v in zip(ks, mu): ax.annotate(f(v, 2), (k, v), xytext=(0, 8), textcoords='offset points', ha='center', fontsize=9)
ax.axhline(0, color='#555', lw=.8)
ax.set_xticks(ks); ax.set_xlabel('warehouses in training'); ax.set_ylabel('R² on a warehouse outside training')
ax.set_title('Model performance versus the number of training warehouses')
import matplotlib.ticker as mt; ax.yaxis.set_major_locator(mt.MultipleLocator(.05)); U.comma_axis(ax, 'y', 2)
plt.show()
U.note(f"With 3 warehouses in training R² = {f(mu[0], 2)}, with 9 it is {f(mu[-1], 2)}. Growth is uneven and the spread between "
       f"repeats is large (band on the chart): ten warehouses are too few to evaluate the model confidently. New warehouses "
       f"are needed both for performance and for a reliable evaluation.")

## 6. Link with POPs pesticide sites

9 of the 10 XRF warehouses coincide with POPs sites on the chemicals.tj portal. Six of them have DDT and HCH analyses,
which makes it possible to compare metal and pesticide contamination at the same sites.

In [ ]:
lk = U.DATA / 'source_data/chemicals_tj/pop_layer/metals_link'
sl = pd.read_csv(lk / 'site_level.csv', encoding='utf-8-sig')
summ = json.loads((lk / 'summary.json').read_text(encoding='utf-8'))
t = pd.DataFrame({'XRF warehouse': sl.xrf_site.map(name_en), 'POPs site': sl.tj_code, 'Metals priority': sl.metals_priority_rank,
                  'Pb, XRF median, mg/kg': sl.xrf_Pb_median, 'POPs samples': sl.pop_samples,
                  'DDT+HCH, maximum, mg/kg': sl.pop_burden_max_mgkg}).sort_values('Metals priority')
U.table(t, nd=1)
s_ = summ['site_level_spearman']
U.note(f"Rank correlation between metals priority and pesticide contamination over {s_['n']} sites: "
       f"{f(s_['rho'], 2)} (p = {f(s_['p'], 2)}); a site-level relationship is not confirmed.")

At the site level the relationship is weak, but at three warehouses the most contaminated pesticide samples lie next
to the points of maximum lead (J. Balkhi, Dusti, Kushoniyon). This is a reason for joint sampling: XRF and laboratory
pesticide samples at the same points. If the link is confirmed, inexpensive field XRF surveys will indicate where to
take costly laboratory samples.

## Conclusions

- Lead is above the MPC at almost half of the points, but above the international residential level only at a few;
  arsenic and mercury need laboratory verification because of the XRF detection limit.
- Warehouse priority is computed transparently and does not depend on the choice of standard.
- The within-warehouse model gives a moderate gain in choosing sampling locations, validated on unseen warehouses.
- The main limitation is 10 warehouses: with 9 in training performance is higher than with fewer, but the spread is
  large. Every new warehouse improves both performance and the reliability of the evaluation.